# HPX Interactome Knowledge Graph

This notebook is a walkthrough of a reproducible proteomics-to-knowledge-graph workflow built from mzIdentML files associated with the PRIDE dataset **PXD060789**.

## Project goal
To identify proteins associated with HPX conditions, compare them against transferrin and control conditions, and organize the resulting proteins into an interpretable graph linking:

**Condition → Protein → Biological Process**

## Main result
The workflow identifies an HPX-enriched subnetwork centered on **LRP1**, with additional proteins linked to uptake, iron homeostasis, heme handling, complement, coagulation, protease regulation, and lipoprotein biology.


In [ ]:
from pathlib import Path
import pandas as pd

DATA = Path("../data/processed")
META = Path("../data/metadata")
FIGS = Path("../figures")

psm = pd.read_csv(DATA / "psm_table.csv")
protein_summary = pd.read_csv(DATA / "protein_evidence_summary.csv")
annotated = pd.read_csv(DATA / "protein_evidence_annotated.csv")
comparison = pd.read_csv(DATA / "protein_condition_comparison.csv")
top_hpx = pd.read_csv(DATA / "top_hpx_candidates.csv")
proc_ann = pd.read_csv(META / "protein_process_annotations.csv")
graph_table = pd.read_csv(DATA / "protein_process_graph_table.csv")
nodes = pd.read_csv(DATA / "process_graph_nodes.csv")
edges = pd.read_csv(DATA / "process_graph_edges.csv")

print("Loaded all major project tables.")


## 1. Parsed mzID output

The project begins by parsing 12 mzIdentML files into peptide-spectrum match (PSM) and protein-level evidence tables.


In [ ]:
print("PSM rows:", len(psm))
print("Protein summary rows:", len(protein_summary))
print("Annotated protein rows:", len(annotated))
print("Unique accessions:", annotated["protein_accession_clean"].nunique())
print("Unique genes:", annotated["gene_symbol"].dropna().nunique())

psm.head()


In [ ]:
protein_summary.sort_values("n_peptides", ascending=False).head(15)


## 2. HPX candidate proteins

Proteins were compared across three condition classes:
- **HPX**
- **Tf**
- **control**

The comparison table flags proteins that are present in HPX and absent from controls, and calculates a simple HPX-over-background score.


In [ ]:
top_hpx.head(20)


### Interpretation

A few proteins stand out immediately:

- **LRP1** is the strongest HPX-specific anchor in this analysis.
- **TFRC** is shared between HPX and transferrin conditions, which is biologically sensible for uptake-related comparison.
- **CP, HP, HPR, SERPINA1, A2M** add extracellular and heme/iron-related context.
- **C1QB, C1QC, FCN3, F12** support complement and coagulation-linked modules.


## 3. Protein-to-process annotation

Top HPX-associated proteins were assigned small, interpretable biological process labels through a semi-manual curation workflow, producing a simple knowledge graph layer.


In [ ]:
proc_ann.head(20)


In [ ]:
proc_ann["process_group"].value_counts()


These process groups provide the backbone of the knowledge graph:

- uptake
- iron homeostasis
- heme handling
- complement
- coagulation
- protease regulation
- lipoprotein biology


## 4. Graph-ready tables

The final graph links condition classes to proteins and proteins to curated biological processes.


In [ ]:
print("Node count:", len(nodes))
print("Edge count:", len(edges))
print("\nNode types:")
print(nodes["node_type"].value_counts())

nodes.head(10)


In [ ]:
edges.head(15)


## 5. Final figure

Below is the publication-style figure generated from the processed graph tables.


In [ ]:
from IPython.display import Image, display
display(Image(filename=str(FIGS / "figure1_hpx_knowledge_graph.png")))


## 6. Key proteins summary


In [ ]:
focus_genes = ["LRP1", "TFRC", "CP", "HP", "HPR", "A2M", "SERPINA1", "F12", "C1QB", "C1QC", "FCN3"]
comparison.loc[comparison["gene_symbol"].isin(focus_genes)].sort_values(
    ["candidate_HPX_over_background_score", "HPX"],
    ascending=[False, False]
)


## Take-home message

This project demonstrates a reproducible workflow for:

- parsing mzIdentML proteomics outputs in Python
- summarizing protein-level evidence across conditions
- annotating proteins with UniProt-derived identifiers and names
- comparing HPX, transferrin, and control conditions
- building a small knowledge graph linking proteins to biological processes
- generating interactive and publication-style visual outputs

The main biological result is an **HPX-enriched subnetwork centered on LRP1**, with linked uptake, iron homeostasis, heme handling, complement, coagulation, protease regulation, and lipoprotein-associated proteins.
